In [1]:
import pandas as pd
import numpy as np 
from scipy import stats as stat
from scipy.stats import bootstrap as bootstrap
import matplotlib.pyplot as plt
import seaborn as sns
from seaborn import catplot
import random as rd
import os
from sklearn.preprocessing import StandardScaler as ss
from sklearn.decomposition import PCA as pca
import math
from scipy.stats import linregress
import mne
%matplotlib qt
import os
from itertools import compress

import mne
from mne import datasets
import numpy as np
from nilearn.plotting import plot_design_matrix

import mne_nirs
import mne_bids
# Import StatsModels
import statsmodels.formula.api as smf
import matplotlib as mpl

# Import MNE processing
from mne.preprocessing.nirs import beer_lambert_law, optical_density

# Import MNE-BIDS processing
from mne_bids import BIDSPath, get_entity_vals, read_raw_bids,stats

from mne_nirs.channels import get_long_channels, get_short_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.io.fold import fold_channel_specificity

# Import MNE-NIRS processing
from mne_nirs.statistics import run_glm, statsmodels_to_results
from mne_nirs.visualisation import plot_glm_group_topo, plot_glm_surface_projection 
import os.path as op

from mne.datasets.testing import data_path
from mne.viz import set_3d_view

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.ar_model import AutoReg
import statsmodels.api as sm
# Import common libraries
from collections import defaultdict
from copy import deepcopy
from itertools import compress
from pprint import pprint

# Import Plotting Library
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Import StatsModels
import statsmodels.formula.api as smf
from mne import Epochs, events_from_annotations, set_log_level
from mne.preprocessing.nirs import (
    beer_lambert_law,
    optical_density,
    scalp_coupling_index,
    temporal_derivative_distribution_repair,
)

# Import MNE processing
from mne.viz import plot_compare_evokeds

# Import MNE-BIDS processing
from mne_bids import BIDSPath, read_raw_bids

# Import MNE-NIRS processing
from mne_nirs.channels import get_long_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.signal_enhancement import enhance_negative_correlation

C:\Users\pkell\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [38]:
fname = r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Kid Relmatch\kid_rel_match_snirf\2026-05-10_002.snirf"
raw_intensity = mne.io.read_raw_snirf(fname, verbose = True, preload = True)
# raw_intensity.annotations.set_durations(1)
print(raw_intensity)
print("\nSampling frequency:", raw_intensity.info["sfreq"])
print("Recording duration:", raw_intensity.times[-1], "seconds")

Loading C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Kid Relmatch\kid_rel_match_snirf\2026-05-10_002.snirf
Reading 0 ... 3153  =      0.000 ...   619.880 secs...
<RawSNIRF | 2026-05-10_002.snirf, 92 x 3154 (619.9 s), ~2.4 MB, data loaded>

Sampling frequency: 5.0864699898270604
Recording duration: 619.8797999999999 seconds


In [25]:
## plot every 10s on x axis 

raw_intensity.plot(duration = 500, n_channels=len(raw_intensity.ch_names));

In [61]:
# ============================================================
# EXTRACT ORDERED TRIGGERS FROM ALL SNIRF FILES
# ============================================================

from pathlib import Path

import mne
import numpy as np
import pandas as pd


# ============================================================
# USER SETTINGS
# ============================================================

# Folder containing participant SNIRF files.
# Files may be directly inside this folder or in subfolders.
data_dir = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Kid Relmatch\kid_rel_match_snirf"
)

# Output folder
output_dir = data_dir / "trigger_duration_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Trigger labels to retain
wanted_trigger_labels = {
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "stim_channel1",
    "stim_channel2",
    "stim_channel3",
    "stim_channel4",
    "stim_channel5",
    "stim_channel6",
}

# Convert possible MNE annotation names to NIRS Toolbox names
trigger_name_map = {
    "1": "stim_channel1",
    "2": "stim_channel2",
    "3": "stim_channel3",
    "4": "stim_channel4",
    "5": "stim_channel5",
    "6": "stim_channel6",
    "stim_channel1": "stim_channel1",
    "stim_channel2": "stim_channel2",
    "stim_channel3": "stim_channel3",
    "stim_channel4": "stim_channel4",
    "stim_channel5": "stim_channel5",
    "stim_channel6": "stim_channel6",
}

# Set this to True only when the interval from a retained trigger
# to the next retained trigger is the duration you want MATLAB to use.
#
# IMPORTANT:
# This assumes the next retained trigger marks the end of the current
# condition/block. If other events occur between block onset and offset,
# the interval may not be the true task duration.
USE_NEXT_RETAINED_TRIGGER_AS_DURATION = True

# # Optional minimum and maximum acceptable durations.
# # Rows outside this range are flagged but not automatically deleted.
# MIN_EXPECTED_DURATION_SECONDS = 0.10
# MAX_EXPECTED_DURATION_SECONDS = 300.00


# ============================================================
# FIND ALL SNIRF FILES
# ============================================================

snirf_files = sorted(data_dir.rglob("*.snirf"))

if len(snirf_files) == 0:
    raise FileNotFoundError(
        f"No .snirf files were found in:\n{data_dir}"
    )

print("=" * 70)
print("SNIRF TRIGGER EXTRACTION")
print("=" * 70)
print(f"Data folder: {data_dir}")
print(f"SNIRF files found: {len(snirf_files)}")
print(f"Output folder: {output_dir}")


# ============================================================
# STORAGE FOR COMBINED RESULTS
# ============================================================

all_participant_tables = []
processing_log_rows = []


# ============================================================
# PROCESS EACH PARTICIPANT
# ============================================================

for file_number, snirf_file in enumerate(snirf_files, start=1):

    # The participant identifier is taken from the SNIRF filename
    participant_id = snirf_file.stem

    print("\n" + "-" * 70)
    print(
        f"Processing file {file_number} of {len(snirf_files)}: "
        f"{participant_id}"
    )
    print(f"File: {snirf_file}")

    try:
        # ----------------------------------------------------
        # Load raw intensity data
        # ----------------------------------------------------

        raw_intensity = mne.io.read_raw_snirf(
            snirf_file,
            preload=False,
            verbose=False,
        )

        sfreq = float(raw_intensity.info["sfreq"])
        recording_duration = float(raw_intensity.times[-1])

        n_annotations = len(raw_intensity.annotations)

        if n_annotations == 0:
            raise ValueError(
                "The file contains no MNE annotations/triggers."
            )

        # ----------------------------------------------------
        # Convert annotations to MNE events
        # ----------------------------------------------------

        events, event_id = mne.events_from_annotations(
            raw_intensity,
            verbose=False,
        )

        if len(events) == 0:
            raise ValueError(
                "No events were created from the annotations."
            )

        # Convert event samples to seconds relative to recording start
        event_times = (
            events[:, 0] - raw_intensity.first_samp
        ) / sfreq

        event_codes = events[:, 2].astype(int)

        # Reverse event_id:
        # annotation name -> numeric code
        # becomes:
        # numeric code -> annotation name
        code_to_name = {
            int(code): str(name)
            for name, code in event_id.items()
        }

        event_names = [
            code_to_name.get(
                int(code),
                f"Unknown_{int(code)}",
            )
            for code in event_codes
        ]

        # ----------------------------------------------------
        # Create complete event table
        # ----------------------------------------------------

        event_table = pd.DataFrame({
            "original_event_number": np.arange(
                1,
                len(events) + 1,
            ),
            "original_event_name": event_names,
            "event_code": event_codes,
            "onset_seconds": event_times,
        })

        # Time to the next event of any kind
        event_table["time_to_next_any_event_seconds"] = np.append(
            np.diff(event_times),
            np.nan,
        )

        # ----------------------------------------------------
        # Keep the desired experimental triggers
        # ----------------------------------------------------

        retained = event_table.loc[
            event_table["original_event_name"].isin(
                wanted_trigger_labels
            )
        ].copy()

        if retained.empty:
            available_names = sorted(
                event_table["original_event_name"]
                .astype(str)
                .unique()
                .tolist()
            )

            raise ValueError(
                "None of the requested triggers were found. "
                f"Available event names were: {available_names}"
            )

        retained = retained.reset_index(drop=True)

        # Chronological sequence number among retained triggers
        retained["event_order"] = np.arange(
            1,
            len(retained) + 1,
        )

        # MATLAB/NIRS Toolbox-compatible trigger name
        retained["trigger_name"] = (
            retained["original_event_name"]
            .map(trigger_name_map)
        )

        # Time to next retained trigger
        retained[
            "time_to_next_retained_trigger_seconds"
        ] = np.append(
            np.diff(retained["onset_seconds"].to_numpy()),
            np.nan,
        )

        # ----------------------------------------------------
        # Add stored annotation duration when possible
        # ----------------------------------------------------

        # The events array does not preserve duration, so match each
        # retained event back to the closest annotation by description
        # and onset.
        annotation_table = pd.DataFrame({
            "annotation_name": (
                raw_intensity.annotations.description.astype(str)
            ),
            "annotation_onset_seconds": (
                raw_intensity.annotations.onset.astype(float)
            ),
            "stored_annotation_duration_seconds": (
                raw_intensity.annotations.duration.astype(float)
            ),
        })

        stored_durations = []

        for row in retained.itertuples(index=False):

            possible_annotation_names = {
                str(row.original_event_name),
                str(row.trigger_name),
            }

            # Also include the numeric form, such as "1"
            reverse_numeric_map = {
                "stim_channel1": "1",
                "stim_channel2": "2",
                "stim_channel3": "3",
                "stim_channel4": "4",
                "stim_channel6": "6",
            }

            if row.trigger_name in reverse_numeric_map:
                possible_annotation_names.add(
                    reverse_numeric_map[row.trigger_name]
                )

            candidates = annotation_table.loc[
                annotation_table["annotation_name"].isin(
                    possible_annotation_names
                )
            ].copy()

            if candidates.empty:
                stored_durations.append(np.nan)
                continue

            onset_differences = np.abs(
                candidates["annotation_onset_seconds"]
                - float(row.onset_seconds)
            )

            closest_index = onset_differences.idxmin()
            closest_difference = onset_differences.loc[
                closest_index
            ]

            # Allow one sample of timing difference
            onset_tolerance = max(
                1.0 / sfreq,
                0.01,
            )

            if closest_difference <= onset_tolerance:
                stored_duration = candidates.loc[
                    closest_index,
                    "stored_annotation_duration_seconds",
                ]
                stored_durations.append(
                    float(stored_duration)
                )
            else:
                stored_durations.append(np.nan)

        retained[
            "stored_annotation_duration_seconds"
        ] = stored_durations

        # ----------------------------------------------------
        # Select the duration for MATLAB
        # ----------------------------------------------------

        if USE_NEXT_RETAINED_TRIGGER_AS_DURATION:

            retained["duration_seconds"] = retained[
                "time_to_next_retained_trigger_seconds"
            ]

            retained["duration_source"] = (
                "time_to_next_retained_trigger"
            )

        else:

            retained["duration_seconds"] = retained[
                "stored_annotation_duration_seconds"
            ]

            retained["duration_source"] = (
                "stored_mne_annotation_duration"
            )

        # The last event has no next event, so its interval is unknown
        retained["is_last_retained_event"] = (
            retained["event_order"] == retained["event_order"].max()
        )

        # ----------------------------------------------------
        # Validate durations
        # ----------------------------------------------------

        retained["duration_is_missing"] = (
            retained["duration_seconds"].isna()
        )

        retained["duration_is_nonpositive"] = (
            retained["duration_seconds"].notna()
            & (retained["duration_seconds"] <= 0)
        )

        retained["duration_below_expected_minimum"] = (
            retained["duration_seconds"].notna()
            & (
                retained["duration_seconds"]
                < MIN_EXPECTED_DURATION_SECONDS
            )
        )

        retained["duration_above_expected_maximum"] = (
            retained["duration_seconds"].notna()
            & (
                retained["duration_seconds"]
                > MAX_EXPECTED_DURATION_SECONDS
            )
        )

        retained["duration_flag"] = np.select(
            [
                retained["duration_is_missing"],
                retained["duration_is_nonpositive"],
                retained["duration_below_expected_minimum"],
                retained["duration_above_expected_maximum"],
            ],
            [
                "missing",
                "nonpositive",
                "below_expected_minimum",
                "above_expected_maximum",
            ],
            default="ok",
        )

        # ----------------------------------------------------
        # Add participant and file information
        # ----------------------------------------------------

        retained.insert(
            0,
            "subject",
            participant_id,
        )

        retained.insert(
            1,
            "source_file",
            snirf_file.name,
        )

        retained.insert(
            2,
            "source_path",
            str(snirf_file),
        )

        retained.insert(
            3,
            "sampling_frequency_hz",
            sfreq,
        )

        retained.insert(
            4,
            "recording_duration_seconds",
            recording_duration,
        )

        # ----------------------------------------------------
        # Arrange MATLAB-friendly columns
        # ----------------------------------------------------

        output_columns = [
            "subject",
            "event_order",
            "trigger_name",
            "onset_seconds",
            "duration_seconds",
            "duration_source",
            "duration_flag",
            "original_event_number",
            "original_event_name",
            "event_code",
            "time_to_next_any_event_seconds",
            "time_to_next_retained_trigger_seconds",
            "stored_annotation_duration_seconds",
            "is_last_retained_event",
            "duration_is_missing",
            "duration_is_nonpositive",
            "duration_below_expected_minimum",
            "duration_above_expected_maximum",
            "sampling_frequency_hz",
            "recording_duration_seconds",
            "source_file",
            "source_path",
        ]

        retained = retained[output_columns]

        # ----------------------------------------------------
        # Save participant-level CSV
        # ----------------------------------------------------

        participant_output_file = (
            output_dir
            / f"{participant_id}_trigger_sequence.csv"
        )

        retained.to_csv(
            participant_output_file,
            index=False,
            float_format="%.6f",
        )

        # Add to combined output
        all_participant_tables.append(retained)

        n_valid_durations = int(
            (retained["duration_flag"] == "ok").sum()
        )

        n_missing_durations = int(
            retained["duration_is_missing"].sum()
        )

        processing_log_rows.append({
            "subject": participant_id,
            "source_file": snirf_file.name,
            "status": "success",
            "error_message": "",
            "n_annotations": n_annotations,
            "n_all_events": len(event_table),
            "n_retained_events": len(retained),
            "n_valid_durations": n_valid_durations,
            "n_missing_durations": n_missing_durations,
            "sampling_frequency_hz": sfreq,
            "recording_duration_seconds": recording_duration,
        })

        print(f"Retained triggers: {len(retained)}")
        print(f"Valid durations: {n_valid_durations}")
        print(f"Missing durations: {n_missing_durations}")
        print(f"Saved: {participant_output_file.name}")

    except Exception as error:

        print(f"ERROR: {error}")

        processing_log_rows.append({
            "subject": participant_id,
            "source_file": snirf_file.name,
            "status": "failed",
            "error_message": str(error),
            "n_annotations": np.nan,
            "n_all_events": np.nan,
            "n_retained_events": np.nan,
            "n_valid_durations": np.nan,
            "n_missing_durations": np.nan,
            "sampling_frequency_hz": np.nan,
            "recording_duration_seconds": np.nan,
        })


# ============================================================
# COMBINE ALL PARTICIPANT TABLES
# ============================================================

if len(all_participant_tables) == 0:
    raise RuntimeError(
        "No participant files were processed successfully. "
        "Check the processing log and trigger names."
    )

combined_trigger_table = pd.concat(
    all_participant_tables,
    ignore_index=True,
)

combined_trigger_table = combined_trigger_table.sort_values(
    by=[
        "subject",
        "event_order",
    ]
).reset_index(drop=True)


# ============================================================
# CREATE MATLAB IMPORT TABLE
# ============================================================

# This compact table contains the columns most useful when assigning
# participant- and event-specific durations in MATLAB.
matlab_trigger_table = combined_trigger_table[
    [
        "subject",
        "event_order",
        "trigger_name",
        "onset_seconds",
        "duration_seconds",
        "duration_flag",
        "source_file",
    ]
].copy()

# Rename trigger_name to make its purpose explicit.
# Leave it as trigger_name if that is what your MATLAB code expects.
matlab_trigger_table["subject"] = (
    matlab_trigger_table["subject"].astype(str)
)

matlab_trigger_table["trigger_name"] = (
    matlab_trigger_table["trigger_name"].astype(str)
)


# ============================================================
# SAVE COMBINED OUTPUT FILES
# ============================================================

combined_output_file = (
    output_dir
    / "all_subjects_complete_trigger_sequence.csv"
)

matlab_output_file = (
    output_dir
    / "all_subjects_matlab_trigger_durations.csv"
)

processing_log_file = (
    output_dir
    / "trigger_extraction_processing_log.csv"
)

combined_trigger_table.to_csv(
    combined_output_file,
    index=False,
    float_format="%.6f",
)

matlab_trigger_table.to_csv(
    matlab_output_file,
    index=False,
    float_format="%.6f",
)

processing_log = pd.DataFrame(
    processing_log_rows
)

processing_log.to_csv(
    processing_log_file,
    index=False,
    float_format="%.6f",
)


# ============================================================
# PRINT SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("TRIGGER EXTRACTION COMPLETE")
print("=" * 70)

print(
    f"Files found: "
    f"{len(snirf_files)}"
)

print(
    f"Files processed successfully: "
    f"{(processing_log['status'] == 'success').sum()}"
)

print(
    f"Files that failed: "
    f"{(processing_log['status'] == 'failed').sum()}"
)

print(
    f"Total retained trigger events: "
    f"{len(combined_trigger_table)}"
)

print(
    f"Events with valid durations: "
    f"{(combined_trigger_table['duration_flag'] == 'ok').sum()}"
)

print(
    f"Events with missing durations: "
    f"{combined_trigger_table['duration_is_missing'].sum()}"
)

print("\nMATLAB-friendly duration file:")
print(matlab_output_file)

print("\nComplete trigger information file:")
print(combined_output_file)

print("\nProcessing log:")
print(processing_log_file)


# ============================================================
# DISPLAY FIRST ROWS
# ============================================================

print("\nFirst 30 rows of the MATLAB-friendly table:")
print(
    matlab_trigger_table.head(30).to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

print("\nProcessing summary:")
print(
    processing_log.to_string(
        index=False,
    )
)

SNIRF TRIGGER EXTRACTION
Data folder: C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Kid Relmatch\kid_rel_match_snirf
SNIRF files found: 25
Output folder: C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Kid Relmatch\kid_rel_match_snirf\trigger_duration_outputs

----------------------------------------------------------------------
Processing file 1 of 25: 2026-04-26_001
File: C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Kid Relmatch\kid_rel_match_snirf\2026-04-26_001.snirf
Retained triggers: 40
Valid durations: 39
Missing durations: 1
Saved: 2026-04-26_001_trigger_sequence.csv

----------------------------------------------------------------------
Processing file 2 of 25: 2026-05-02_001
File: C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Kid Relmatch\kid_rel_match_snirf\2026-05-02_001.snirf
Retained triggers: 42
Valid durations: 41
Missing durations: 1
Saved: 2026-05-02_001_trigger_sequence.csv

----------------------------------------------------------------------
Pr

In [63]:
# ============================================================
# EXTRACT PARTICIPANT-SPECIFIC TASK DURATIONS FROM ALL SNIRF FILES
#
# Duration definition:
#     task duration = onset of next stim_channel5
#                     minus onset of current task trigger
#
# stim_channel5 is used only as the rest-onset boundary.
# It is NOT exported as a modeled task condition for MATLAB.
# ============================================================

from pathlib import Path

import mne
import numpy as np
import pandas as pd


# ============================================================
# USER SETTINGS
# ============================================================

# Folder containing all participant .snirf files.
# Files may be directly in this folder or inside subfolders.
data_dir = Path(
    r"C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data"
    r"\Kid Relmatch\kid_rel_match_snirf"
)

# Output folder
output_dir = data_dir / "trigger_duration_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Experimental task conditions to export to MATLAB
task_trigger_names = {
    "stim_channel1",
    "stim_channel2",
    "stim_channel3",
    "stim_channel4",
}

# Rest-onset marker used to end each task block
rest_trigger_name = "stim_channel5"

# Practice condition.
# Set to True only when you want stim_channel6 included in the
# MATLAB stimulus-duration table and modeled in the GLM.
INCLUDE_STIM_CHANNEL6 = False

if INCLUDE_STIM_CHANNEL6:
    task_trigger_names.add("stim_channel6")

# Expected duration limits used for quality-control flags.
# These do not automatically remove events.
MIN_EXPECTED_TASK_DURATION_SECONDS = 5.0
MAX_EXPECTED_TASK_DURATION_SECONDS = 40.0

# Require the next rest marker to occur before another task trigger.
# If another task trigger occurs first, the sequence is flagged.
REQUIRE_REST_BEFORE_NEXT_TASK = True


# ============================================================
# NORMALIZE MNE ANNOTATION NAMES TO NIRS TOOLBOX NAMES
# ============================================================

trigger_name_map = {
    "1": "stim_channel1",
    "2": "stim_channel2",
    "3": "stim_channel3",
    "4": "stim_channel4",
    "5": "stim_channel5",
    "6": "stim_channel6",
    "stim_channel1": "stim_channel1",
    "stim_channel2": "stim_channel2",
    "stim_channel3": "stim_channel3",
    "stim_channel4": "stim_channel4",
    "stim_channel5": "stim_channel5",
    "stim_channel6": "stim_channel6",
}


# ============================================================
# FIND ALL SNIRF FILES
# ============================================================

snirf_files = sorted(data_dir.rglob("*.snirf"))

if len(snirf_files) == 0:
    raise FileNotFoundError(
        f"No .snirf files were found in:\n{data_dir}"
    )

print("=" * 80)
print("PARTICIPANT-SPECIFIC FNIRS TASK-DURATION EXTRACTION")
print("=" * 80)
print(f"Data directory: {data_dir}")
print(f"SNIRF files found: {len(snirf_files)}")
print(f"Output directory: {output_dir}")
print(f"Task triggers: {sorted(task_trigger_names)}")
print(f"Rest trigger: {rest_trigger_name}")


# ============================================================
# STORAGE FOR ALL PARTICIPANTS
# ============================================================

all_task_tables = []
all_complete_event_tables = []
processing_log_rows = []


# ============================================================
# PROCESS EACH PARTICIPANT
# ============================================================

for file_number, snirf_file in enumerate(snirf_files, start=1):

    participant_id = snirf_file.stem

    print("\n" + "-" * 80)
    print(
        f"Processing {file_number} of {len(snirf_files)}: "
        f"{participant_id}"
    )
    print(f"File: {snirf_file}")

    try:

        # ----------------------------------------------------
        # Load SNIRF data
        # ----------------------------------------------------

        raw_intensity = mne.io.read_raw_snirf(
            snirf_file,
            preload=False,
            verbose=False,
        )

        sfreq = float(raw_intensity.info["sfreq"])
        recording_duration = float(raw_intensity.times[-1])
        n_annotations = len(raw_intensity.annotations)

        if n_annotations == 0:
            raise ValueError(
                "The file contains no annotations or triggers."
            )

        # ----------------------------------------------------
        # Convert annotations to MNE events
        # ----------------------------------------------------

        events, event_id = mne.events_from_annotations(
            raw_intensity,
            verbose=False,
        )

        if len(events) == 0:
            raise ValueError(
                "No events could be generated from the annotations."
            )

        event_times = (
            events[:, 0] - raw_intensity.first_samp
        ) / sfreq

        event_codes = events[:, 2].astype(int)

        # Convert numeric event code back to annotation label
        code_to_name = {
            int(code): str(name)
            for name, code in event_id.items()
        }

        original_event_names = [
            code_to_name.get(
                int(code),
                f"Unknown_{int(code)}",
            )
            for code in event_codes
        ]

        # ----------------------------------------------------
        # Build a complete event table
        # ----------------------------------------------------

        event_table = pd.DataFrame({
            "subject": participant_id,
            "source_file": snirf_file.name,
            "source_path": str(snirf_file),
            "original_event_number": np.arange(
                1,
                len(events) + 1,
            ),
            "original_event_name": original_event_names,
            "event_code": event_codes,
            "onset_seconds": event_times,
        })

        # Normalize annotation names to MATLAB/NIRS names
        event_table["trigger_name"] = (
            event_table["original_event_name"]
            .map(trigger_name_map)
        )

        # Mark any annotation names that were not recognized
        event_table["trigger_recognized"] = (
            event_table["trigger_name"].notna()
        )

        event_table[
            "time_to_next_any_event_seconds"
        ] = np.append(
            np.diff(event_times),
            np.nan,
        )

        all_complete_event_tables.append(event_table.copy())

        # ----------------------------------------------------
        # Confirm required rest marker exists
        # ----------------------------------------------------

        rest_events = event_table.loc[
            event_table["trigger_name"] == rest_trigger_name
        ].sort_values("onset_seconds")

        if rest_events.empty:
            raise ValueError(
                f"No {rest_trigger_name} rest-onset markers were found."
            )

        # ----------------------------------------------------
        # Select task events only
        # ----------------------------------------------------

        task_events = event_table.loc[
            event_table["trigger_name"].isin(task_trigger_names)
        ].copy()

        task_events = task_events.sort_values(
            "onset_seconds"
        ).reset_index(drop=True)

        if task_events.empty:
            raise ValueError(
                "No modeled task triggers were found."
            )

        # Add chronological task-event order across all task types
        task_events["event_order"] = np.arange(
            1,
            len(task_events) + 1,
        )

        # Add occurrence number separately within each trigger type
        task_events["trigger_occurrence"] = (
            task_events
            .groupby("trigger_name")
            .cumcount()
            + 1
        )

        # ----------------------------------------------------
        # Calculate each task event's duration
        # ----------------------------------------------------

        duration_rows = []

        for task_row in task_events.itertuples(index=False):

            task_onset = float(task_row.onset_seconds)
            task_name = str(task_row.trigger_name)

            # All later rest-onset markers
            later_rest_events = rest_events.loc[
                rest_events["onset_seconds"] > task_onset
            ].sort_values("onset_seconds")

            # All later task triggers
            later_task_events = task_events.loc[
                task_events["onset_seconds"] > task_onset
            ].sort_values("onset_seconds")

            if later_rest_events.empty:

                rest_onset = np.nan
                duration_seconds = np.nan
                next_rest_event_number = np.nan
                duration_flag = "missing_next_rest"

            else:

                next_rest = later_rest_events.iloc[0]

                rest_onset = float(
                    next_rest["onset_seconds"]
                )

                next_rest_event_number = int(
                    next_rest["original_event_number"]
                )

                duration_seconds = (
                    rest_onset - task_onset
                )

                # --------------------------------------------
                # Validate the event sequence
                # --------------------------------------------

                another_task_before_rest = False
                next_task_onset = np.nan
                next_task_name = ""

                if not later_task_events.empty:

                    next_task = later_task_events.iloc[0]

                    next_task_onset = float(
                        next_task["onset_seconds"]
                    )

                    next_task_name = str(
                        next_task["trigger_name"]
                    )

                    another_task_before_rest = (
                        next_task_onset < rest_onset
                    )

                if not np.isfinite(duration_seconds):
                    duration_flag = "missing_duration"

                elif duration_seconds <= 0:
                    duration_flag = "nonpositive"

                elif (
                    REQUIRE_REST_BEFORE_NEXT_TASK
                    and another_task_before_rest
                ):
                    duration_flag = "another_task_before_rest"

                elif (
                    duration_seconds
                    < MIN_EXPECTED_TASK_DURATION_SECONDS
                ):
                    duration_flag = "implausibly_short"

                elif (
                    duration_seconds
                    > MAX_EXPECTED_TASK_DURATION_SECONDS
                ):
                    duration_flag = "implausibly_long"

                else:
                    duration_flag = "ok"

            duration_rows.append({
                "subject": participant_id,
                "event_order": int(task_row.event_order),
                "trigger_name": task_name,
                "trigger_occurrence": int(
                    task_row.trigger_occurrence
                ),
                "onset_seconds": task_onset,
                "rest_onset_seconds": rest_onset,
                "duration_seconds": duration_seconds,
                "duration_flag": duration_flag,
                "task_original_event_number": int(
                    task_row.original_event_number
                ),
                "rest_original_event_number": (
                    next_rest_event_number
                ),
                "sampling_frequency_hz": sfreq,
                "recording_duration_seconds": (
                    recording_duration
                ),
                "source_file": snirf_file.name,
                "source_path": str(snirf_file),
            })

        participant_task_table = pd.DataFrame(
            duration_rows
        )

        # ----------------------------------------------------
        # Save participant-level MATLAB-compatible table
        # ----------------------------------------------------

        participant_output_file = (
            output_dir
            / f"{participant_id}_matlab_task_durations.csv"
        )

        participant_task_table.to_csv(
            participant_output_file,
            index=False,
            float_format="%.6f",
        )

        all_task_tables.append(participant_task_table)

        # ----------------------------------------------------
        # Participant-level validation report
        # ----------------------------------------------------

        duration_counts = (
            participant_task_table["duration_flag"]
            .value_counts(dropna=False)
            .to_dict()
        )

        n_ok = int(
            (
                participant_task_table["duration_flag"]
                == "ok"
            ).sum()
        )

        n_problem = int(
            len(participant_task_table) - n_ok
        )

        processing_log_rows.append({
            "subject": participant_id,
            "source_file": snirf_file.name,
            "status": "success",
            "error_message": "",
            "n_annotations": n_annotations,
            "n_all_events": len(event_table),
            "n_task_events": len(participant_task_table),
            "n_rest_events": len(rest_events),
            "n_ok_durations": n_ok,
            "n_flagged_durations": n_problem,
            "duration_flag_counts": str(duration_counts),
            "sampling_frequency_hz": sfreq,
            "recording_duration_seconds": (
                recording_duration
            ),
        })

        print(f"Task events: {len(participant_task_table)}")
        print(f"Rest markers: {len(rest_events)}")
        print(f"Valid durations: {n_ok}")
        print(f"Flagged durations: {n_problem}")
        print(f"Duration flags: {duration_counts}")
        print(f"Saved: {participant_output_file.name}")

        if n_problem > 0:

            print("\nFlagged task events:")

            print(
                participant_task_table.loc[
                    participant_task_table[
                        "duration_flag"
                    ] != "ok",
                    [
                        "trigger_name",
                        "trigger_occurrence",
                        "onset_seconds",
                        "rest_onset_seconds",
                        "duration_seconds",
                        "duration_flag",
                    ],
                ].to_string(
                    index=False,
                    float_format=lambda value: (
                        f"{value:.4f}"
                    ),
                )
            )

    except Exception as error:

        print(f"ERROR: {error}")

        processing_log_rows.append({
            "subject": participant_id,
            "source_file": snirf_file.name,
            "status": "failed",
            "error_message": str(error),
            "n_annotations": np.nan,
            "n_all_events": np.nan,
            "n_task_events": np.nan,
            "n_rest_events": np.nan,
            "n_ok_durations": np.nan,
            "n_flagged_durations": np.nan,
            "duration_flag_counts": "",
            "sampling_frequency_hz": np.nan,
            "recording_duration_seconds": np.nan,
        })


# ============================================================
# COMBINE ALL PARTICIPANTS
# ============================================================

if len(all_task_tables) == 0:
    raise RuntimeError(
        "No participant task-duration tables were created."
    )

combined_task_table = pd.concat(
    all_task_tables,
    ignore_index=True,
)

combined_task_table = combined_task_table.sort_values(
    [
        "subject",
        "event_order",
    ]
).reset_index(drop=True)


# ============================================================
# CREATE COMPACT MATLAB IMPORT TABLE
# ============================================================

matlab_trigger_table = combined_task_table[
    [
        "subject",
        "event_order",
        "trigger_name",
        "trigger_occurrence",
        "onset_seconds",
        "rest_onset_seconds",
        "duration_seconds",
        "duration_flag",
        "source_file",
    ]
].copy()

matlab_trigger_table["subject"] = (
    matlab_trigger_table["subject"].astype(str)
)

matlab_trigger_table["trigger_name"] = (
    matlab_trigger_table["trigger_name"].astype(str)
)


# ============================================================
# CREATE VALID-ONLY TABLE
# ============================================================

# This table includes only rows that passed all duration checks.
# Use the full table first for inspection before deciding whether
# exclusion of flagged rows is appropriate.
matlab_valid_only_table = matlab_trigger_table.loc[
    matlab_trigger_table["duration_flag"] == "ok"
].copy()


# ============================================================
# COMBINE COMPLETE EVENT TABLES
# ============================================================

combined_complete_event_table = pd.concat(
    all_complete_event_tables,
    ignore_index=True,
)

combined_complete_event_table = (
    combined_complete_event_table
    .sort_values(
        [
            "subject",
            "original_event_number",
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# SAVE OUTPUT FILES
# ============================================================

matlab_output_file = (
    output_dir
    / "all_subjects_matlab_trigger_durations.csv"
)

matlab_valid_output_file = (
    output_dir
    / "all_subjects_matlab_trigger_durations_valid_only.csv"
)

complete_event_output_file = (
    output_dir
    / "all_subjects_complete_event_sequence.csv"
)

processing_log_file = (
    output_dir
    / "trigger_duration_processing_log.csv"
)

flagged_output_file = (
    output_dir
    / "all_flagged_task_durations.csv"
)


matlab_trigger_table.to_csv(
    matlab_output_file,
    index=False,
    float_format="%.6f",
)

matlab_valid_only_table.to_csv(
    matlab_valid_output_file,
    index=False,
    float_format="%.6f",
)

combined_complete_event_table.to_csv(
    complete_event_output_file,
    index=False,
    float_format="%.6f",
)

processing_log = pd.DataFrame(
    processing_log_rows
)

processing_log.to_csv(
    processing_log_file,
    index=False,
    float_format="%.6f",
)

flagged_task_table = matlab_trigger_table.loc[
    matlab_trigger_table["duration_flag"] != "ok"
].copy()

flagged_task_table.to_csv(
    flagged_output_file,
    index=False,
    float_format="%.6f",
)


# ============================================================
# FINAL VALIDATION SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("TRIGGER-DURATION EXTRACTION COMPLETE")
print("=" * 80)

print(f"SNIRF files found: {len(snirf_files)}")

print(
    "Successfully processed: "
    f"{(processing_log['status'] == 'success').sum()}"
)

print(
    "Failed files: "
    f"{(processing_log['status'] == 'failed').sum()}"
)

print(
    f"Total task events exported: "
    f"{len(matlab_trigger_table)}"
)

print("\nDuration flags:")

print(
    matlab_trigger_table[
        "duration_flag"
    ].value_counts(dropna=False)
)

print("\nDuration summary by trigger:")

print(
    matlab_trigger_table
    .groupby("trigger_name")["duration_seconds"]
    .describe()
    .round(4)
)

print("\nEvent counts by subject and trigger:")

print(
    matlab_trigger_table
    .groupby(
        [
            "subject",
            "trigger_name",
        ]
    )
    .size()
    .unstack(fill_value=0)
)

print("\nMATLAB-compatible full duration file:")
print(matlab_output_file)

print("\nMATLAB-compatible valid-only duration file:")
print(matlab_valid_output_file)

print("\nComplete event-sequence file:")
print(complete_event_output_file)

print("\nFlagged-duration file:")
print(flagged_output_file)

print("\nProcessing log:")
print(processing_log_file)


# ============================================================
# DISPLAY FIRST ROWS
# ============================================================

print("\nFirst 30 rows of MATLAB duration table:")

print(
    matlab_trigger_table.head(30).to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

if not flagged_task_table.empty:

    print("\nAll flagged task durations:")

    print(
        flagged_task_table.to_string(
            index=False,
            float_format=lambda value: f"{value:.4f}",
        )
    )
else:

    print("\nNo task durations were flagged.")

PARTICIPANT-SPECIFIC FNIRS TASK-DURATION EXTRACTION
Data directory: C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Kid Relmatch\kid_rel_match_snirf
SNIRF files found: 25
Output directory: C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Kid Relmatch\kid_rel_match_snirf\trigger_duration_outputs
Task triggers: ['stim_channel1', 'stim_channel2', 'stim_channel3', 'stim_channel4']
Rest trigger: stim_channel5

--------------------------------------------------------------------------------
Processing 1 of 25: 2026-04-26_001
File: C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Kid Relmatch\kid_rel_match_snirf\2026-04-26_001.snirf
Task events: 12
Rest markers: 14
Valid durations: 12
Flagged durations: 0
Duration flags: {'ok': 12}
Saved: 2026-04-26_001_matlab_task_durations.csv

--------------------------------------------------------------------------------
Processing 2 of 25: 2026-05-02_001
File: C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\Kid Relmatch\kid_rel_match_snirf\202